#widgets

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("schemaBronze", "bronze")
dbutils.widgets.text("catalogo", "PROJECT_DEV")
dbutils.widgets.text("storageName", "adlssmartprojectdev13")

In [0]:
container = dbutils.widgets.get("container")
schemaBronze = dbutils.widgets.get("schemaBronze")
catalogo = dbutils.widgets.get("catalogo")
storageName = dbutils.widgets.get("storageName")

# definir rutas
path_inicio = f"abfss://{container}@{storageName}.dfs.core.windows.net/raw_data_project/"
path_target = f"{catalogo}.{schemaBronze}."

In [0]:
# importar functions essentials
from pyspark.sql.functions import *
from pyspark.sql.types import *

## ORDER ITEMS

In [0]:
orders_items_schema = StructType(fields=[StructField("order_id", StringType(), False),
                                     StructField("order_item_id", IntegerType(), True),
                                     StructField("product_id", StringType(), True),
                                     StructField("seller_id", StringType(), True),
                                     StructField("shipping_limit_date", TimestampType(), True),
                                     StructField("price", DecimalType(precision=10, scale=2), True),
                                     StructField("freight_value", DecimalType(precision=10, scale=2), True)
])

In [0]:
df_order_items = spark.read.format("csv") \
    .option("header", True) \
    .schema(orders_items_schema) \
    .load(f"{path_inicio}olist_order_items_dataset.csv")

## Order Payments

In [0]:
orders_payments_schema = StructType(fields=[StructField("order_id", StringType(), False),
                                     StructField("payment_sequential", IntegerType(), True),
                                     StructField("payment_type", StringType(), True),
                                     StructField("payment_installments", IntegerType(), True),
                                     StructField("payment_value", DecimalType(precision=10, scale=2), True)
])

In [0]:
df_order_payments = spark.read.format("csv") \
    .option("header", True) \
    .schema(orders_payments_schema) \
    .load(f"{path_inicio}olist_order_payments_dataset.csv")

## Orders Reviews

In [0]:
# leer el csv
df_order_review = spark.read.format("csv") \
    .option("header", True) \
    .load(f"{path_inicio}olist_order_reviews_dataset.csv")

In [0]:
## se hace asi porqeu las columnas estan en diferente posicion y solo se necesitan algunas
df_order_review = df_order_review.select( 
    "order_id",
    "review_score",
    "review_creation_date",
    "review_answer_timestamp"
)
## LIMPIEZA A ESTA TABLA YA QUE TIENE VALORES NULOS

##Orders

In [0]:
orders_schema = StructType(fields=[StructField("order_id", StringType(), False),
                                     StructField("customer_id", StringType(), True),
                                     StructField("order_status", StringType(), True),
                                     StructField("order_purchase_timestamp", TimestampType(), True),
                                     StructField("order_approved_at", TimestampType(), True),
                                     StructField("order_delivered_carrier_date", TimestampType(), True),
                                     StructField("order_delivered_customer_date", TimestampType(), True),
                                     StructField("order_estimated_delivery_date", TimestampType(), True)
])

In [0]:
# leer el csv
df_orders = spark.read.format("csv") \
    .option("header", True) \
    .schema(orders_schema) \
    .load(f"{path_inicio}olist_orders_dataset.csv")

## cargar la informacion en los targets

In [0]:
df_orders.write.mode("overwrite").insertInto(f"{path_target}orders")

In [0]:
df_order_payments.write.mode("overwrite").insertInto(f"{path_target}orders_payments")

In [0]:
df_order_review.write.mode("overwrite").insertInto(f"{path_target}orders_review")

In [0]:
df_order_items.write.mode("overwrite").insertInto(f"{path_target}orders_items")

##aplicar optimize y vacum

In [0]:
%sql
---- aplicar optimizer a la tabla target
OPTIMIZE ${catalogo}.${esquema_target}.orders
ZORDER BY order_id;

OPTIMIZE ${catalogo}.${esquema_target}.orders_payments
ZORDER BY order_id;

OPTIMIZE ${catalogo}.${esquema_target}.orders_review
ZORDER BY order_id;

OPTIMIZE ${catalogo}.${esquema_target}.orders_items
ZORDER BY order_id;


In [0]:
%sql
--- habilitar vacum dinamico
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

In [0]:
%sql
---- solo tener archivos inferiores  24 horas
VACUUM ${catalogo}.${esquema_target}.orders RETAIN 24 HOURS ;

VACUUM ${catalogo}.${esquema_target}.orders_payments RETAIN 24 HOURS;

VACUUM ${catalogo}.${esquema_target}.orders_review RETAIN 24 HOURS;

VACUUM ${catalogo}.${esquema_target}.orders_items RETAIN 24 HOURS;